# BridgeLink ASL — WLASL-100 Training (Google Colab)

End-to-end pipeline: download WLASL-100 → extract MediaPipe Holistic
landmarks → train a lightweight Transformer classifier → export model.

## Before running

1. **Runtime → Change runtime type → T4 GPU** (free tier is fine).
2. You will need a **Kaggle account** with a valid API token (`kaggle.json`).
   - Go to kaggle.com → Your Profile → Account → Create New Token → downloads `kaggle.json`.
   - You'll upload it when prompted below.
3. Total runtime ≈ 60 minutes (landmark extraction is the slow part).
4. Outputs are saved to your **Google Drive** under `BridgeLink-ASL/` so they survive runtime resets.


## 1. Setup

In [ ]:
!pip install -q mediapipe kaggle

import os, json, random, math, time
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Mount Google Drive

All outputs are saved to Drive so you don't lose them if Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Project folder on your Drive
PROJECT_DIR = Path("/content/drive/MyDrive/BridgeLink-ASL")
PROJECT_DIR.mkdir(exist_ok=True, parents=True)
(PROJECT_DIR / "models").mkdir(exist_ok=True)
(PROJECT_DIR / "results").mkdir(exist_ok=True)
(PROJECT_DIR / "landmarks").mkdir(exist_ok=True)

# Local working directory (fast SSD on the Colab VM)
LOCAL = Path("/content/wlasl")
LOCAL.mkdir(exist_ok=True)

print("Drive project dir:", PROJECT_DIR)
print("Local working dir:", LOCAL)


## 3. Download the WLASL dataset from Kaggle

When prompted, **upload your `kaggle.json`** file. This is your Kaggle API token.

If you don't have one: kaggle.com → Your Profile icon → Settings → API → Create New Token.


In [ ]:
from google.colab import files
import shutil

# Upload kaggle.json
print("Upload your kaggle.json file:")
uploaded = files.upload()

# Set up Kaggle credentials
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
shutil.move("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
print("Kaggle credentials configured.")


In [ ]:
# Download the dataset (~10 GB, takes 3-5 min)
DATASET_DIR = LOCAL / "wlasl-processed"

if not DATASET_DIR.exists():
    print("Downloading WLASL dataset... this takes 3-5 minutes.")
    !kaggle datasets download -d risangbaskoro/wlasl-processed -p {LOCAL} --unzip
    # The unzip creates a folder structure — find where it landed
    print("Download complete.")
else:
    print("Dataset already downloaded.")

print("Contents:", [p.name for p in LOCAL.iterdir()])


## 4. Locate the dataset files

In [ ]:
def find_wlasl_assets(root):
    root = Path(root)
    json_candidates = list(root.rglob("WLASL_v0.3.json")) + list(root.rglob("wlasl_v0.3.json"))
    if not json_candidates:
        # Try case-insensitive
        json_candidates = [p for p in root.rglob("*.json") if "wlasl" in p.name.lower() and "v0" in p.name.lower()]
    if not json_candidates:
        raise FileNotFoundError(f"WLASL JSON not found under {root}. Check download.")
    json_path = json_candidates[0]
    for candidate in [json_path.parent / "videos", json_path.parent, json_path.parent.parent / "videos"]:
        if candidate.exists() and any(candidate.glob("*.mp4")):
            return json_path, candidate
    for d in root.rglob("*"):
        if d.is_dir() and any(d.glob("*.mp4")):
            return json_path, d
    raise FileNotFoundError("Could not locate WLASL mp4 videos.")

JSON_PATH, VIDEO_DIR = find_wlasl_assets(LOCAL)
print("JSON :", JSON_PATH)
print("Videos:", VIDEO_DIR)
num_videos = len(list(VIDEO_DIR.glob('*.mp4')))
print(f"Total mp4 files: {num_videos}")
print("Sample:", [p.name for p in list(VIDEO_DIR.glob('*.mp4'))[:5]])


## 5. Parse WLASL JSON → filter to WLASL-100

In [ ]:
with open(JSON_PATH) as f:
    raw = json.load(f)

all_samples = []
for entry in raw:
    gloss = entry["gloss"]
    for inst in entry["instances"]:
        # Filter to WLASL-100 subset
        subsets = inst.get("subsets", [])
        # Some mirrors use different key names — handle both
        if not subsets:
            subsets = inst.get("subset", [])
        if isinstance(subsets, str):
            subsets = [subsets]
        if "WLASL100" not in subsets and "asl100" not in [s.lower() for s in subsets]:
            continue
        video_id = inst["video_id"]
        video_path = VIDEO_DIR / f"{video_id}.mp4"
        if not video_path.exists():
            continue
        all_samples.append({
            "gloss": gloss,
            "video_id": video_id,
            "video_path": str(video_path),
            "split": inst.get("split", "train"),
            "frame_start": inst.get("frame_start", 1),
            "frame_end": inst.get("frame_end", -1),
            "bbox": inst.get("bbox", None),
        })

print(f"WLASL-100 instances with available videos: {len(all_samples)}")

# FALLBACK: if subset tags are missing, take the top-100 glosses by frequency
if len(all_samples) < 50:
    print("WARNING: subset tags not found — falling back to top-100 glosses by frequency.")
    all_samples_full = []
    for entry in raw:
        gloss = entry["gloss"]
        for inst in entry["instances"]:
            video_id = inst["video_id"]
            video_path = VIDEO_DIR / f"{video_id}.mp4"
            if not video_path.exists():
                continue
            all_samples_full.append({
                "gloss": gloss,
                "video_id": video_id,
                "video_path": str(video_path),
                "split": inst.get("split", "train"),
                "frame_start": inst.get("frame_start", 1),
                "frame_end": inst.get("frame_end", -1),
                "bbox": inst.get("bbox", None),
            })
    # Keep top 100 glosses
    gloss_counts = Counter(s["gloss"] for s in all_samples_full)
    top100 = {g for g, _ in gloss_counts.most_common(100)}
    all_samples = [s for s in all_samples_full if s["gloss"] in top100]
    print(f"After top-100 fallback: {len(all_samples)} instances")

split_counts = Counter(s["split"] for s in all_samples)
class_counts = Counter(s["gloss"] for s in all_samples)
print(f"Split distribution: {dict(split_counts)}")
print(f"Classes present: {len(class_counts)}")
print(f"Median samples/class: {int(np.median(list(class_counts.values())))}")

# If no split info, create random 80/10/10
if len(split_counts) <= 1:
    print("No split info found — creating random 80/10/10 split.")
    random.shuffle(all_samples)
    n = len(all_samples)
    for i, s in enumerate(all_samples):
        if i < int(0.8 * n):
            s["split"] = "train"
        elif i < int(0.9 * n):
            s["split"] = "val"
        else:
            s["split"] = "test"
    split_counts = Counter(s["split"] for s in all_samples)
    print(f"New split distribution: {dict(split_counts)}")


## 6. Dataset visualizations

In [ ]:
RESULTS = PROJECT_DIR / "results"

# Class distribution
top_classes = class_counts.most_common(30)
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar([c[0] for c in top_classes], [c[1] for c in top_classes], color="#4c72b0")
ax.set_title("WLASL-100 — top 30 classes by instance count")
ax.set_ylabel("# clips")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.savefig(RESULTS / "class_distribution.png", dpi=150)
plt.show()

# Split distribution
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(split_counts.keys(), split_counts.values(), color=["#4c72b0", "#55a868", "#c44e52"])
ax.set_title("Train / val / test split")
ax.set_ylabel("# clips")
plt.tight_layout()
plt.savefig(RESULTS / "split_distribution.png", dpi=150)
plt.show()


In [ ]:
# Preview sample frames
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, sample in zip(axes, random.sample(all_samples, min(5, len(all_samples)))):
    cap = cv2.VideoCapture(sample["video_path"])
    ok, frame = cap.read()
    cap.release()
    if ok:
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(sample["gloss"], fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.savefig(RESULTS / "sample_frames.png", dpi=150)
plt.show()


## 7. MediaPipe Holistic landmark extraction

Converts each clip into a `(32, 225)` float32 array.

**This is the slow step — ~30–45 min.** Progress bar will update. Landmarks are cached to Drive so you never redo this.

In [ ]:
import mediapipe as mp
mp_holistic = mp.solutions.holistic

SEQ_LEN = 32
FEAT_DIM = 21*3 + 21*3 + 33*3  # 225

def flatten_lm(landmarks, n):
    if landmarks is None:
        return np.zeros(n * 3, dtype=np.float32)
    return np.array([[p.x, p.y, p.z] for p in landmarks.landmark], dtype=np.float32).flatten()

def extract_clip(video_path, frame_start, frame_end, bbox=None, seq_len=SEQ_LEN):
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        cap.release()
        return None
    start = max(0, frame_start - 1)
    end = total if frame_end == -1 else min(total, frame_end)
    end = max(start + 1, end)
    frames = []
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    for _ in range(end - start):
        ok, frame = cap.read()
        if not ok:
            break
        if bbox is not None:
            x1, y1, x2, y2 = [int(v) for v in bbox]
            h, w = frame.shape[:2]
            x1, y1, x2, y2 = max(0,x1), max(0,y1), min(w,x2), min(h,y2)
            if x2 > x1 and y2 > y1:
                frame = frame[y1:y2, x1:x2]
        frames.append(frame)
    cap.release()
    if not frames:
        return None
    idx = np.linspace(0, len(frames) - 1, seq_len).astype(int)
    sampled = [frames[i] for i in idx]
    seq = np.zeros((seq_len, FEAT_DIM), dtype=np.float32)
    with mp_holistic.Holistic(static_image_mode=False, model_complexity=1,
                              min_detection_confidence=0.3, min_tracking_confidence=0.3) as holistic:
        for t, f in enumerate(sampled):
            rgb = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
            res = holistic.process(rgb)
            lh = flatten_lm(res.left_hand_landmarks, 21)
            rh = flatten_lm(res.right_hand_landmarks, 21)
            pose = flatten_lm(res.pose_landmarks, 33)
            seq[t] = np.concatenate([lh, rh, pose])
    return seq


In [ ]:
from tqdm.auto import tqdm

LANDMARK_DIR = PROJECT_DIR / "landmarks"
LANDMARK_DIR.mkdir(exist_ok=True)

cached = extracted = failed = 0
manifest = []

for sample in tqdm(all_samples, desc="Extracting landmarks"):
    out_path = LANDMARK_DIR / f"{sample['video_id']}.npy"
    if out_path.exists():
        cached += 1
        manifest.append({**sample, "landmark_path": str(out_path)})
        continue
    try:
        seq = extract_clip(sample["video_path"], sample["frame_start"],
                           sample["frame_end"], sample.get("bbox"))
        if seq is None:
            failed += 1
            continue
        np.save(out_path, seq)
        manifest.append({**sample, "landmark_path": str(out_path)})
        extracted += 1
    except Exception as e:
        failed += 1
        if failed <= 5:
            print(f"  fail {sample['video_id']}: {e}")

print(f"\nExtracted: {extracted}  Cached: {cached}  Failed: {failed}")
print(f"Usable samples: {len(manifest)}")

with open(PROJECT_DIR / "landmarks_manifest.json", "w") as f:
    json.dump(manifest, f)


## 8. PyTorch dataset with augmentation

In [ ]:
glosses = sorted({m['gloss'] for m in manifest})
label_map = {g: i for i, g in enumerate(glosses)}
inv_label_map = {i: g for g, i in label_map.items()}
print(f"Classes: {len(label_map)}")

train_samples = [m for m in manifest if m['split'] == 'train']
val_samples   = [m for m in manifest if m['split'] == 'val']
test_samples  = [m for m in manifest if m['split'] == 'test']
print(f"Train: {len(train_samples)}  Val: {len(val_samples)}  Test: {len(test_samples)}")

class LandmarkSignDataset(Dataset):
    def __init__(self, samples, label_map, augment=False):
        self.samples = samples
        self.label_map = label_map
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        s = self.samples[i]
        x = np.load(s['landmark_path'])
        y = self.label_map[s['gloss']]
        if self.augment:
            x = self._augment(x)
        return torch.from_numpy(x.astype(np.float32)), y

    def _augment(self, x):
        x = x + np.random.normal(0, 0.01, x.shape).astype(np.float32)
        if random.random() < 0.5:
            x = x.copy()
            x[:, 0::3] = 1.0 - x[:, 0::3]
            lh = x[:, 0:63].copy()
            rh = x[:, 63:126].copy()
            x[:, 0:63] = rh
            x[:, 63:126] = lh
        if random.random() < 0.3:
            start = random.randint(0, max(0, x.shape[0] - 4))
            x = x.copy()
            x[start:start+3] = 0.0
        return x

train_ds = LandmarkSignDataset(train_samples, label_map, augment=True)
val_ds   = LandmarkSignDataset(val_samples, label_map, augment=False)
test_ds  = LandmarkSignDataset(test_samples, label_map, augment=False)

BATCH = 64
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2)
print(f"Batches — train: {len(train_loader)}  val: {len(val_loader)}  test: {len(test_loader)}")


## 9. Sign Transformer model (~1.3M params)

In [ ]:
class SignTransformer(nn.Module):
    def __init__(self, num_classes, d_model=192, nhead=4, layers=4,
                 seq_len=32, feat_dim=225, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(feat_dim, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, seq_len + 1, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)
        self.config = dict(num_classes=num_classes, d_model=d_model, nhead=nhead,
                           layers=layers, seq_len=seq_len, feat_dim=feat_dim, dropout=dropout)

    def forward(self, x):
        B = x.size(0)
        h = self.input_proj(x)
        cls = self.cls_token.expand(B, -1, -1)
        h = torch.cat([cls, h], dim=1) + self.pos_embed
        h = self.encoder(h)
        h = self.norm(h[:, 0])
        return self.head(h)

model = SignTransformer(num_classes=len(label_map)).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {n_params/1e6:.2f}M")


## 10. Training loop

Best model is saved to Google Drive after every improvement.

In [ ]:
EPOCHS = 60
LR = 3e-4
WEIGHT_DECAY = 1e-2

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

history = {"train_loss": [], "val_acc": [], "val_top5": []}
best_val = 0.0
best_epoch = -1
best_path = PROJECT_DIR / "models/sign_transformer_best.pt"

def evaluate(loader):
    model.eval()
    correct1 = correct5 = total = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            top5 = logits.topk(min(5, logits.size(1)), dim=1).indices
            pred = logits.argmax(dim=1)
            correct1 += (pred == y).sum().item()
            correct5 += (top5 == y.unsqueeze(1)).any(dim=1).sum().item()
            total += y.size(0)
            all_preds.extend(pred.cpu().tolist())
            all_labels.extend(y.cpu().tolist())
    return correct1/max(total,1), correct5/max(total,1), all_preds, all_labels

print("Training...")
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = loss_fn(logits, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running += loss.item() * y.size(0)
    scheduler.step()
    train_loss = running / max(len(train_ds), 1)

    val_top1, val_top5, _, _ = evaluate(val_loader)
    history["train_loss"].append(train_loss)
    history["val_acc"].append(val_top1)
    history["val_top5"].append(val_top5)
    marker = ""
    if val_top1 > best_val:
        best_val = val_top1
        best_epoch = epoch
        torch.save({
            "state_dict": model.state_dict(),
            "label_map": label_map,
            "config": model.config,
        }, best_path)
        marker = "  ← best"
    if epoch % 5 == 0 or marker:
        print(f"epoch {epoch:02d}  loss={train_loss:.3f}  val_top1={val_top1:.3f}  val_top5={val_top5:.3f}{marker}")

print(f"\nBest val top-1: {best_val:.3f} at epoch {best_epoch}")


## 11. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"])
axes[0].set_title("Training loss"); axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss")
axes[1].plot(history["val_acc"], label="top-1")
axes[1].plot(history["val_top5"], label="top-5")
axes[1].set_title("Validation accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
plt.savefig(PROJECT_DIR / "results/training_curves.png", dpi=150)
plt.show()


## 12. Final test evaluation

In [ ]:
ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["state_dict"])

test_top1, test_top5, preds, labels = evaluate(test_loader)
print(f"TEST  top-1: {test_top1:.3f}   top-5: {test_top5:.3f}")

try:
    from sklearn.metrics import classification_report, confusion_matrix
    present_labels = sorted(set(labels + preds))
    target_names = [inv_label_map[i] for i in present_labels]
    report_txt = classification_report(labels, preds, labels=present_labels,
                                       target_names=target_names, zero_division=0, digits=3)
    print(report_txt)
    with open(PROJECT_DIR / "results/classification_report.txt", "w") as f:
        f.write(report_txt)
    cm = confusion_matrix(labels, preds, labels=list(range(len(label_map))))
except ImportError:
    print("sklearn not found — computing confusion matrix manually")
    cm = np.zeros((len(label_map), len(label_map)), dtype=int)
    for p, l in zip(preds, labels):
        cm[l, p] += 1

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(cm, cmap="Blues")
ax.set_title(f"Confusion matrix (test) — top-1 {test_top1:.1%}")
ax.set_xlabel("predicted"); ax.set_ylabel("true")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(PROJECT_DIR / "results/confusion_matrix.png", dpi=150)
plt.show()

metrics = {
    "val_top1_best": round(best_val, 4),
    "val_top1_best_epoch": best_epoch,
    "test_top1": round(test_top1, 4),
    "test_top5": round(test_top5, 4),
    "num_classes": len(label_map),
    "train_samples": len(train_samples),
    "val_samples": len(val_samples),
    "test_samples": len(test_samples),
    "model_params_M": round(n_params / 1e6, 2),
    "epochs": EPOCHS,
}
with open(PROJECT_DIR / "results/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))


## 13. Export

In [ ]:
with open(PROJECT_DIR / "models/labels.json", "w") as f:
    json.dump({
        "label_map": label_map,
        "inv_label_map": {str(i): g for i, g in inv_label_map.items()}
    }, f, indent=2)

print("\nSaved to Google Drive at:", PROJECT_DIR)
print()
print("Models:")
for p in sorted((PROJECT_DIR / "models").iterdir()):
    print(f"  {p.name}  ({p.stat().st_size/1e6:.2f} MB)")
print()
print("Results:")
for p in sorted((PROJECT_DIR / "results").iterdir()):
    print(f"  {p.name}")
print()
print("=" * 60)
print("DONE! Next steps:")
print("1. Go to Google Drive → BridgeLink-ASL → models/")
print("   Download sign_transformer_best.pt and labels.json")
print("2. Go to Google Drive → BridgeLink-ASL → results/")
print("   Download all the PNGs and metrics.json for your report")
print("3. Drop the model files into your repo and run the app")
print("=" * 60)


## What you just built

- **Data**: WLASL-100 filtered subset
- **Features**: MediaPipe Holistic → 225-d landmark vectors × 32 frames per clip
- **Model**: 4-layer Transformer encoder, ~1.3M parameters, CLS-token classification
- **Training**: 60 epochs, cosine schedule, label smoothing, gradient clipping, augmentation (coord jitter, horizontal flip with hand swap, random temporal dropout)
- **Expected results**: top-1 ≈ 40–60%, top-5 ≈ 65–85%

All outputs are in your Google Drive under `BridgeLink-ASL/`. They will survive if this Colab session dies.
